In [1]:
import json
import re
from extraction import Extraction
from embeddings import Embeddings

In [2]:
extraction = Extraction()
embeddings = Embeddings()

In [3]:
with open("config.json", "r") as f:
    CONFIG = json.load(f)

In [4]:
CONFIG["Format"]["Question"]["General"] # "General": "Please answer this question: {question}"
CONFIG["Format"]["Question"]["Factual"] # "Factual": "Please answer this question with a factual approach: {question}"
CONFIG["Format"]["Question"]["Embedding"] # "Embedding": "Please answer this question with an embedding approach: {question}"

def classify_question(question: str) -> tuple[str, str]:
    """
    Classifies a question into one of three types (general, factual, or embedding)
    and extracts the pure question without the format prefix.
    
    Args:
        question (str): The input question with format prefix
        
    Returns:
        tuple[str, str]: (pure_question, question_type)
        - pure_question: The question without the format prefix
        - question_type: One of 'general', 'factual', or 'embedding'
    """
    def format_to_regex(fmt: str) -> re.Pattern:
        # Escape the format string but keep the {question} placeholder
        escaped = re.escape(fmt)
        # Replace the escaped {question} with a capturing group
        pattern = escaped.replace(r'\{question\}', r'(.+)')
        return re.compile(f"^{pattern}$")
    
    # Create patterns from CONFIG formats
    patterns = {
        question_type.lower(): format_to_regex(fmt)
        for question_type, fmt in CONFIG["Format"]["Question"].items()
    }
    
    # Try to match each pattern
    for question_type, pattern in patterns.items():
        match = pattern.match(question)
        if match:
            return match.group(1), question_type
            
    # If no pattern matches, assume it's a general question
    return question, "general"

In [5]:
def classify_question_simple(question: str) -> tuple[str, str]:
    """
    A simpler version of question classification that looks for keywords and splits on ':'
    
    Args:
        question (str): The input question
        
    Returns:
        tuple[str, str]: (pure_question, question_type)
        - pure_question: The question after the first colon
        - question_type: One of 'general', 'factual', or 'embedding'
    """
    question = question.lower()
    
    # Determine type based on keywords
    if "factual" in question:
        question_type = "factual"
    elif "embedding" in question:
        question_type = "embedding"
    else:
        question_type = "general"
        
    # Extract pure question after the first colon
    if ":" in question:
        pure_question = question.split(":", 1)[1].strip()
    else:
        pure_question = question
        
    return pure_question, question_type

In [6]:
def process_question(question: str) -> str:
    pure_q, q_type = classify_question(question)

    entity = extraction.extract_entity(pure_q)
    entity_label, entity_uri, entity_score, entity_distance = extraction.link_entity(entity)
    
    relation = extraction.extract_relation(pure_q)
    relation_label, relation_uri, relation_score, relation_distance = extraction.link_relation(relation)

    if q_type == "factual" or q_type == "general":

        answer = "This is a default factual answer."  # Placeholder for actual extraction logic
        return f"The factual answer is: {answer}"
    elif q_type == "embedding":

        results = embeddings.get_best_result(
            entity_uri, 
            relation_uri,
            1
        )
        head_uri, head_label, head_score, head_rank = results[0]

        with open("cache/entities/entity_types.json", "r", encoding="utf-8") as f:
            entity_types = json.load(f)
        
        type_qid = entity_types.get(head_uri, "N/A")
        return f"The answer suggested by embeddings is: {head_label} (type: {type_qid})"

In [7]:
questions = {
    "Please answer this question with a factual approach: From what country is the movie 'Aro Tolbukhin. En la mente del asesino'?":
        "The factual answer is: Mexico",

    "Please answer this question with a factual approach: Who is the screenwriter of 'Shortcut to Happiness'?":
        "The factual answer is: Pete Dexter",

    "Please answer this question with a factual approach: Who directed ‘Fargo’?":
        "The factual answer is: Ethan Coen and Joel Coen",

    "Please answer this question with a factual approach: What genre is the movie 'Bandit Queen'?":
        "The factual answer is: drama film and biographical film and crime film",

    "Please answer this question with a factual approach: When did the movie 'Miracles Still Happen' come out?":
        "The factual answer is: 1974-07-19",

    "Please answer this question with an embedding approach: Who is the director of ‘Apocalypse Now’?":
        "The answer suggested by embeddings is: John Milius (type: Q5)",

    "Please answer this question with an embedding approach: Who is the screenwriter of ‘12 Monkeys’?":
        "The answer suggested by embeddings is: Carol Florence (type: Q5)",

    "Please answer this question with an embedding approach: What is the genre of ‘Shoplifters’?":
        "The answer suggested by embeddings is: comedy film (type: Q201658)",

    "Please answer this question: Who is the director of ‘Good Will Hunting’?":
        "The factual answer is: Gus Van Sant. The answer suggested by embeddings is: Harmony Korine (type: Q5)",
}

In [8]:
for question in questions.keys():
    response = process_question(question)
    print(f"Question: {question}")
    print(f"Response: {response}")
    print("-" * 50)

Question: Please answer this question with a factual approach: From what country is the movie 'Aro Tolbukhin. En la mente del asesino'?
Response: The factual answer is: This is a default factual answer.
--------------------------------------------------
Question: Please answer this question with a factual approach: Who is the screenwriter of 'Shortcut to Happiness'?
Response: The factual answer is: This is a default factual answer.
--------------------------------------------------
Question: Please answer this question with a factual approach: Who directed ‘Fargo’?
Response: The factual answer is: This is a default factual answer.
--------------------------------------------------
Question: Please answer this question with a factual approach: What genre is the movie 'Bandit Queen'?
Response: The factual answer is: This is a default factual answer.
--------------------------------------------------
Question: Please answer this question with a factual approach: When did the movie 'Miracl